# Component 3 — Model Comparison Report

This notebook is the **reporting and interpretation layer**, not a second training implementation. Run the four training scripts and `compare_models.py` from the repository root before executing these cells.

The canonical training logic lives in `src/model_research.py`; each model has a dedicated entry point. This avoids maintaining two separate implementations of the same experiment.

**Research question:** do Day 1 or multi-day dietary features improve four-class BP-associated category classification compared with demographic features alone?

**Primary metric:** mean 5-fold cross-validation macro-F1. The held-out test set is reported after selection and is not used to select the model. The experiment is not a clinical diagnosis or validated forecast.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Run this notebook from component3-nutrition/notebooks.
RESULTS_DIR = Path("../results/model_research")
required = [
    RESULTS_DIR / "all_models_cv_results.csv",
    RESULTS_DIR / "all_models_test_results.csv",
    RESULTS_DIR / "selected_candidate.json",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Run all four train_*.py scripts and src/compare_models.py first. Missing: "
        + ", ".join(missing)
    )

cv_results = pd.read_csv(required[0])
test_results = pd.read_csv(required[1])
selected = json.loads(required[2].read_text(encoding="utf-8"))
display(cv_results[[
    "model_label", "feature_set_label", "cv_macro_f1_mean", "cv_macro_f1_std"
]].sort_values("cv_macro_f1_mean", ascending=False))
print("Selected by cross-validation only:")
display(pd.Series(selected))

## Cross-validation comparison

Compare models using mean macro-F1 and its fold-to-fold standard deviation. Do not select a model using the held-out test scores.

In [ ]:
cv_pivot = cv_results.pivot(
    index="model_label", columns="feature_set_label", values="cv_macro_f1_mean"
)
display(cv_pivot)
ax = cv_pivot.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Mean 5-fold CV macro-F1")
ax.set_xlabel("")
ax.set_title("Model and feature-set comparison (training partition CV)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Held-out test results

These metrics describe the chosen split after model selection. Do not use them to choose between models; doing so would turn the test set into a validation set.

In [ ]:
test_all = test_results[test_results["group"] == "all_adults"].copy()
display(test_all[[
    "model_label", "feature_set_label", "accuracy", "macro_precision",
    "macro_recall", "macro_f1", "weighted_f1"
]].sort_values("macro_f1", ascending=False))
test_pivot = test_all.pivot(
    index="model_label", columns="feature_set_label", values="macro_f1"
)
ax = test_pivot.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Held-out test macro-F1")
ax.set_xlabel("")
ax.set_title("Held-out test results (reporting only)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Older-adult subgroup

The age-60+ results are a secondary subgroup analysis using only participants in the held-out test partition. Interpret cautiously because subgroup sample sizes are smaller.

In [ ]:
older = test_results[test_results["group"] == "age_60_plus"].copy()
display(older[[
    "model_label", "feature_set_label", "n", "accuracy", "macro_precision",
    "macro_recall", "macro_f1", "weighted_f1"
]].sort_values("macro_f1", ascending=False))

## Interpretation checklist

- Is the best mean CV macro-F1 materially higher than the demographic-only baseline?
- Is the difference large relative to fold-to-fold variation?
- Which classes have low precision or recall? Inspect the classification report and confusion matrix created by `tune_best_model.py`.
- Does the age-60+ subgroup behave differently, and how uncertain is that estimate?
- Do not claim clinical utility from this internal split alone. External validation and prospective evaluation would be required.
- NHANES diet inputs are observed 24-hour recalls; they are not equivalent to caregiver-planned meals or seven-day dietary histories.